# LEAF & CROP
## Latent encoding of agricultural features and context-based recommendations for optimised planting

TODO: Add information about dataset, idea behind the project, approach, ressources

## Imports and configurations

### Imports of useful libraries

#### Pip installs

In [1]:
!pip install -q python-box
!pip install -q scikit-learn
!pip install -q ax-platform
!pip install -q torch
!pip install -q torchvision
!pip install -q torchmetrics
!pip install -q wandb
!pip install -q PyYAML
!pip install -q pandas
!pip install -q numpy
!pip install -q imbalanced-learn


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip insta

#### Imports

In [ ]:
# Libraries for logging and general utilities
import random
import logging
import wandb
import time
import math
import yaml
from box.box import Box
import functools
import operator
import gc

# Libraries for data manipulation and analysis
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from ax.service.ax_client import AxClient, ObjectiveProperties
from imblearn.over_sampling import SMOTE

# Helpful libraries
from datetime import datetime
from pathlib import Path
from typing import Any, Tuple

# Deep learning libraries
import torch
from torch import nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchmetrics.classification import (MulticlassAccuracy, MulticlassConfusionMatrix,
                                         MulticlassF1Score, MulticlassAUROC)
from torchmetrics.regression import MeanSquaredError, MeanAbsoluteError

### Configuration to control model architecture and training

TODO: Explaination of parameters

In [ ]:
%%writefile Autoencoder.yaml
name: "Autoencoder"
latent_dim: 64
encoder:
    num_layers:2
decoder:
    num_layers:2
classifier:
    num_layers:2
    dropout: 0.3
loss_weights:
    classification: 1.0
    decoder: 1.0

training:
    num_epochs: 220
    # Learning rate parameters
    learning_rate: 0.0003
    learning_rate_scheduler: "cosine"
    learning_rate_min: 0.000001
    # Optimizer parameters
    beta1: 0.9
    beta2: 0.999
    weight_decay: 0.01

TODO: Explaination of general parameters

In [ ]:
%%writefile config.yaml
# Change 'local' to 'colab' if running in Google Colab, don't forget to adapt source path
environment: 'local'
# Seed for deterministics
seed: 42

num_classes: 22
use_model: "Autoencoder"

logging:
    log_interval: 5
    eval_interval: 5
    # For WandB logging
    use_wandb: True
    team_name: "UPV_projects"
    project_name: "LEAF_and_CROP"
    api_key: null
    # Add-on for the run name
    run_name_additional: None

dataset:
    name: "Crop Recommendation"
    batch_size: 64
    # For loading data
    # local path or google drive path, e.g. 'gdrive/MyDrive/Colab Notebooks/leaf_and_crop/dataset'
    root_path: ''
    dataset_filename: 'Crop_recommendation.csv'
    
    # For dataaugmentation
    data_augmentation:
        mixup:
            active: True
            alpha: 0.2
        noise:
            enabled: True,
            noise_level: 0.091
            probability: 0.9
        dropout:
            enabled: True,
            probability: 0.05
        shift:
            enabled: True,
            max_shift: 0.0455
            probability: 0.7

TODO: Explaination of optimization parameters

In [ ]:
%%writefile optimization_config.yaml
active: True

optimization_config:
  objectives:
    classification_accuracy:
      minimize: False
      threshold: 0.90  # Optional: Min tolerated accuracy
  total_trials: 2

search_space:
  - name: "trraining.learning_rate"
    type: "range"
    bounds: [0.000001, 0.01]
    value_type: "float"

In [ ]:
# Transform dict to Box for attribute-style access
with open("config.yaml", "r") as f:
    CONFIG = Box(yaml.safe_load(f))
print(f"General configuration loaded from config.yaml")
with open(f"{CONFIG.use_model}.yaml", "r") as f:
    MODEL_CONFIG = Box(yaml.safe_load(f))
print(f"Model-specific configuration loaded for model {MODEL_CONFIG.name}")
CONFIG.training = MODEL_CONFIG.training
del MODEL_CONFIG.training
CONFIG.model = MODEL_CONFIG
print(f"Configuration merged for model {CONFIG.model.name}")
with open(f"optimization_config.yaml", "r") as f:
    OPTIMIZATION_CONFIG = Box(yaml.safe_load(f))
    CONFIG.optimization = OPTIMIZATION_CONFIG
print(f"Optimization configuration loaded")


### Dictionary for label mapping

In [ ]:
LABEL_MAPPING = {
    'rice': 0,
    'maize': 1,
    'chickpea': 2,
    'kidneybeans': 3,
    'pigeonpeas': 4,
    'mothbeans': 5,
    'mungbean': 6,
    'blackgram': 7,
    'lentil': 8,
    'pomegranate': 9,
    'banana': 10,
    'mango': 11,
    'grapes': 12,
    'watermelon': 13,
    'muskmelon': 14,
    'apple': 15,
    'orange': 16,
    'papaya': 17,
    'coconut': 18,
    'cotton': 19,
    'jute': 20,
    'coffee': 21
}

#### Support of Google Colab

In [ ]:
# In case of running in colab, mount Google Drive
if CONFIG.environment == 'colab':
    from google.colab import drive
    drive.mount('/content/gdrive')
# In case of running locally no action necessary
elif CONFIG.environment == 'local':
    pass
else:
  raise ValueError(f"Invalid environment '{CONFIG.environment}'. Must be one of 'local' or 'colab'.")

## Helper classes
Classes for utility functionalities

### Class for training parameters
Small helper class to organise the parameters for the training, so the Trainer class holds less attributes.
The attributes stored in this helper class can be easily accessed using getter function.

In [ ]:
class TrainingParameters:
    '''
    Class to encapsulate training parameters.
    Contains learning rate, beta values, weight decay, and number of epochs.
    '''

    def __init__(self,
            lr: float,
            lr_min: float,
            lr_scheduler: str,
            beta1: float,
            beta2: float,
            weight_decay: float,
            epochs: int
        ):
        '''
        Args:
            lr (float): Learning rate
            beta1 (float): Beta1 value for optimizer
            beta2 (float): Beta2 value for optimizer
            weight_decay (float): Weight decay for optimizer
            epochs (int): Number of training epochs

        Initializes the training parameters with the given values.
        '''
        self.lr = lr
        self.lr_min = lr_min
        self.lr_scheduler = lr_scheduler
        self.beta1 = beta1
        self.beta2 = beta2
        self.weight_decay = weight_decay
        self.epochs = epochs

    def get_learning_rate(self) -> float:
        '''
        Get the learning rate.
        '''
        return self.lr

    def get_learning_rate_min(self) -> float:
        '''
        Get the minimum learning rate.
        '''
        return self.lr_min

    def get_betas(self) -> tuple[float, float]:
        '''
        Get the beta values.
        '''
        return (self.beta1, self.beta2)

    def get_weight_decay(self) -> float:
        '''
        Get the weight decay.
        '''
        return self.weight_decay

    def get_epochs(self) -> int:
        '''
        Get the number of epochs.
        '''
        return self.epochs

    def get_lr_scheduler(self) -> str:
        '''
        Get lr_schedular
        '''
        return self.lr_scheduler

## Dataset

TODO: General information about the dataset, e.g. results of the analysis

### Dataaugmentation

TODO: Explain which data augmentation techniques are being used and why
TODO: Work on docstrings in this section

In [ ]:
class GaussianNoise:
    '''
    Class to add Gaussian noise to the input signal.
    '''
    def __init__(self, std_range=(0.0, 0.1), p=0.0):
        """
        std: relative noise strength
        p: probability of applying the noise
        """
        self.std_range = std_range
        self.p = p

    def __call__(self, x):
        """
        Args:
            x: input tensor to which Gaussian noise will be added
        
        Returns:
            result: tensor with added Gaussian noise, if applied, otherwise the original tensor
        """
        if random.random() < self.p:
            std = random.uniform(self.std_range[0], self.std_range[1])
            noise = torch.randn_like(x) * std
            # Maximal noise of given level
            noise = torch.clamp(noise, -0.1, 0.1)
            result = x + noise
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class RandomValueDropout:
    '''
    Class to randomly drop (set to zero) one value in the input signal.
    '''
    def __init__(self, p=0.0):
        '''
        p: probability of dropping a value.
        '''
        self.p = p

    def __call__(self, x):
        '''
        Args:
            x (torch.Tensor): Input tensor.
        Returns:
            torch.Tensor: Tensor with one value dropped.
        '''
        if random.random() < self.p:
            len_x = x.shape[1]
            # Randomly select indices to drop
            one_feature = random.randint(0, len_x - 1)
            x[:, one_feature] = 0.0
        return x

In [ ]:
class Shifting:
    '''
    Class to add a shift to the input signal.
    '''
    def __init__(self, shift=0.05, p=0.0):
        """
        shift: relative shift strength
        p: probability of applying the shift
        """
        self.shift = shift
        self.p = p

    def __call__(self, x):
        """
        Args:
            x: input tensor to which shift will be added

        Returns:
            result: tensor with added shift, if applied, otherwise the original tensor
        """
        if random.random() < self.p:
            shift = random.uniform(-self.shift, self.shift)
            # Apply the shift
            result = x + shift
            # Make sure the result is still between 0 and 1
            result = torch.clamp(result, 0.0, 1.0)
            return result
        return x

In [ ]:
class MixUp:
    '''
    Class to apply MixUp data augmentation.
    '''
    def __init__(self, alpha=0.2):
        """
        alpha: parameter for the Beta distribution to sample the mixing coefficient
        """
        self.alpha = alpha
    
    def __call__(self, x, y):
        """
        Args:
            x: input tensor to which MixUp will be applied
            y: corresponding labels for the input tensor

        Returns:
            mixed_x: tensor with MixUp applied, if applied, otherwise the original tensor
            mixed_y: corresponding labels for the mixed tensor, if MixUp applied, otherwise the original labels
        """
        lam = np.random.beta(self.alpha, self.alpha)
        batch_size = x.size(0)
        mixed_x = x.clone()
        mixed_y = y.clone()

        for i in range(batch_size):
            # Find samples of same class to mix with (not sample i itself)
            same_class = (y == y[i]).nonzero(as_tuple=True)[0]
            same_class = same_class[same_class != i]
            if len(same_class) > 0:
                # Apply mixup with a randomly selected sample from the same class
                j = np.random.choice(same_class.cpu().numpy())
                mixed_x[i] = lam * x[i] + (1 - lam) * x[j]
                mixed_y[i] = lam * y[i] + (1 - lam) * y[j]
        
        return mixed_x, mixed_y

In [ ]:
class DataAugmentation:
    """
    Class for randomly applying data augmentations of the following list:
        - Noise
        - Null masking of RSS values
        - Bias shifting of RSS values
    """
    def __init__(self, cfg: any):
        '''
        Initializes the DataAugmentation with a composition of random
        transformations.
        '''
        self.transform = transforms.Compose([
            GaussianNoise(std_range=(0.0, cfg.data_augmentation.noise.noise_level), p=cfg.data_augmentation.noise.probability),
            Shifting(shift=cfg.data_augmentation.shift.max_shift, p=cfg.data_augmentation.shift.probability),
            RandomValueDropout(p=cfg.data_augmentation.dropout.probability)
        ])

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        '''
        Args:
            x (torch.Tensor): Input tensor to be augmented.
        Returns:
            torch.Tensor: Augmented tensor.

        Method to apply the random data augmentations to the input signal.
        '''
        return self.transform(x)

### Dataset

TODO: Explain loading and preprocessing of data

In [ ]:
class CropRecommendation(Dataset):
    """
    Dataset class for loading the crop recommendation dataset, preprocessing it, 
    and providing samples and labels for training, validation and evaluation.
    """
    def __init__(self, cfg: Any,
                 split: str,
                 sample_transform: transforms = None,
                 label_transform: transforms = None,
                 scaler: MinMaxScaler = None
        ) -> None:
        """
        Initialize the dataset.

        Args:
            cfg: Configuration object with dataset paths and parameters.
            split: One of 'training', 'validation', or 'test'.
            sample_transform: Optional transform to apply to samples.
            label_transform: Optional transform to apply to labels.
        """
        # Retrieve seed to ensure deterministic splitting of data into subsets
        self.random_seed = cfg.seed
        # Store dataset parameters
        self.cfg = cfg.dataset
        # Which subset is supposed to be created
        self.split = split

        self.scaler = scaler

        # Set data root path to access datafile
        self.data_root = Path(self.cfg.root_path)

        # Set up data augmentation
        self.sample_transform = sample_transform
        self.label_transform = label_transform

        # Access dataset file
        self.dataset_path = self.data_root / self.cfg.dataset_filename

        # Check if file exists
        if not self.dataset_path.exists():
            raise FileNotFoundError(f"Dataset file not found at {self.dataset_path}")

        # Load data and preprocess, check whether split is empty
        self.samples, self.labels = self._load_data()

        if len(self.samples) == 0:
            raise RuntimeError(
                f"Dataset split '{split}' is empty. "
            )


    def _load_data(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Load raw data file, preprocess the data and split into subsets.

        Returns:
            Tuple of (samples, labels).
        """
        # Remove unnecessary columns
        drop_columns = []

        # Read data from file
        data = pd.read_csv(self.dataset_path)

        # Split the data into training, validation, and test sets with same distribution of classes in each subset
        training_data, temp_data = train_test_split(
            data,
            # 20% for temp (validation + test)
            test_size=0.20,
            random_state=self.random_seed,
            stratify=data['label']
        )

        validation_data, test_data = train_test_split(
            temp_data,
            # 50% of temp for validation and test
            test_size=0.5,
            random_state=self.random_seed,
            stratify=data['label']
        )

        if self.split == 'training':
            data = training_data
        elif self.split == 'validation':
            data = validation_data
        elif self.split == 'test':
            data = test_data
        else:
            raise ValueError(f"Invalid split '{self.split}'. Must be one of 'training', 'validation', or 'test'.")
        
        # Extract labels
        label_columns = ['label']
        labels = data.loc[:, label_columns].copy()
        samples = data.drop(columns=label_columns)
        
        # Normalize
        if self.split == 'training':
            # Fit the scaler on the training data and transform it
            samples = self.scaler.fit_transform(samples)
        else:
            samples = self.scaler.transform(samples)

        # Use SMOTE to generate more samples for each class to increase the size of the dataset and balance the classes
        # Each class should have at least 200 samples
        unique_classes = labels.unique()
        strategy = {y: 200 for y in unique_classes}

        # Apply SMOTE to the training data only
        if self.split == 'training':
            smote = SMOTE(random_state=self.random_seed, sampling_strategy=strategy)
            samples, labels = smote.fit_resample(samples, labels)
        
        labels = labels.replace(LABEL_MAPPING)

        return np.array(samples, dtype=np.float32), np.array(labels, dtype=np.float32)


    def __len__(self) -> int:
        """
        Return the number of samples in the dataset.
        """
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Union[int, torch.Tensor]]:
        """
        Return a sample and its label.

        Output shapes:
            - Sample: (C,) where C is the number of features
            - Label: num_classes,) for multi-label classification
        """
        x = torch.from_numpy(self.samples[idx])
        y = torch.from_numpy(self.labels[idx])

        if self.sample_transform:
            x = self.sample_transform(x)
        if self.label_transform:
            y = self.label_transform(y)

        return x, y


## Class for general utilities

TODO: Explain different utility functions

In [ ]:
class Utilities:
    '''
    Class to encapsulate utilities for training, such as logging.
    '''

    def __init__(self):
        '''
        Class to organize different utility functionalities.
        '''
        pass

    def seed_worker(self, worker_id):
        '''
        Seed worker for DataLoader to ensure reproducibility.
        '''
        _ = worker_id  # Unused, but required by the DataLoader
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    def set_by_path(self, nested_box, path, value):
        '''
        Set a value in a nested Box using a dot-separated path. Used to adapt the configuration
        for optimization.

        Args:
            nested_box (Box): The Box object to modify.
            path (str): Dot-separated path to the value to set (e.g., "training.learning_rate").
            value: The value to set at the specified path.
        '''
        keys = path.split('.')
        functools.reduce(operator.getitem, keys[:-1], nested_box)[keys[-1]] = value

    def get_dataset(self, cfg, split='training', scaler=None):
        '''
        utility function to get dataset and dataloader for a given split.

        Args:
            cfg: Config object
            split: 'training', 'validation', or 'test'

        Return:
            dataset
            Dataloader
        '''
        if split == 'training':
            transform_ops = [
                DataAugmentation(cfg.dataset)
            ]
        # No augmentation for test or validation
        else:
            transform_ops = [
            ]

        datapoint_transform = transforms.Compose(transform_ops)
        label_transform = transforms.Compose([])

        dataset = CropRecommendation(cfg, split, sample_transform=datapoint_transform,
                        label_transform=label_transform, scaler=scaler)
        
        # Get current fitted scaler, important to normalize validation & test set
        current_scaler = getattr(dataset, 'scaler', None)


        # Generator for reproducibility
        g = torch.Generator()
        g.manual_seed(cfg.seed)

        dataloader = DataLoader(
            dataset,
            batch_size=cfg.dataset.batch_size,
            shuffle=(split == 'training'), # Just shuffle during training
            worker_init_fn=self.seed_worker,
            generator=g
        )

        return dataset, dataloader, current_scaler


    def set_seed(self, seed: int):
        '''
        Set random seed for reproducibility across all libraries.

        Args:
            seed (int): Random seed value
        '''
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    def get_parameters_table(self, model: torch.nn.Module, position: int = 0) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize
            position (int): The position in the model hierarchy (recursive depth)

        Returns a list of tuples containing (module name, position, number of trainable parameters).
        '''
        # name, position, number trainable parameters
        lines = []
        num_trainable_params = 0
        for _, param in model.named_parameters(recurse=False):
            if param.requires_grad:
                num_trainable_params += math.prod(param.size())

        child_lines = []
        for module in model.children():
            child_lines += self.get_parameters_table(module, position + 1)

        sum_params = 0
        for line in child_lines:
            if line[1] == position + 1:
                sum_params += line[2]

        if num_trainable_params == 0:
            lines.append([type(model).__name__, position, sum_params])
        else:
            lines.append([type(model).__name__, position, num_trainable_params])
        lines += child_lines

        return lines

    def get_model_summary(self, model: torch.nn.Module) -> str:
        '''
        Args:
            model (torch.nn.Module): The PyTorch model to summarize

        Returns a string representation of the model's summary.
        '''
        lines = self.get_parameters_table(model)
        summary = ""
        for line in lines:
            summary += "  " * line[1] + f"└─ {line[0]}: {line[2]} trainable parameters\n"
        summary += "=================================================================\n"
        summary += f"Total Trainable Parameters: {lines[0][2]}\n"
        return summary

    def save_best_parameters(self, best_params: Box, filename: str = "best_parameters.yaml"):
        '''
        Save the best parameters to a YAML file.

        Args:
            best_params (Dict[str, Any]): Dictionary containing the best parameters to save.
            filename (str): The name of the YAML file to save the parameters to.
        '''

        with open(filename, "w") as f:
            yaml.dump(best_params.to_dict(), f)

## Evaluator class
Class for handling the evaluation of the model with the test set, calculating different relevant metrics

TODO: Explain why these metrics and how it is done

In [ ]:
class Evaluator:
    """
    Evaluator for binary and multiclass classification tasks.

    Handles model evaluation with metrics computation for both:
        - Binary classification: num_classes=1 (BCELoss) or num_classes=2 (CrossEntropyLoss)
        - Multiclass classification: num_classes>2 (CrossEntropyLoss)

    Automatically selects metrics (Accuracy, Precision, Recall, F1, AUROC, Confusion Matrix)
    based on the classification task type.
    """

    def __init__(
        self,
        cfg: Any,
        eval_loader: DataLoader,
        model: nn.Module,
        device: torch.device,
        wb_run: Any

    ):
        """
        Args:
            cfg: Hydra config. Must contain 'num_classes'.
            eval_loader (DataLoader): DataLoader for the evaluation dataset.
            model (nn.Module): The model to evaluate.
            device (torch.device): 'cuda' or 'cpu'.
            criterion (nn.Module, optional): Loss function (nn.BCELoss or nn.CrossEntropyLoss).
                                            Uses 'nn.CrossEntropyLoss' as default.
        """
        self.cfg = cfg
        self.eval_loader = eval_loader
        self.model = model
        self.device = device
        self.wb_run = wb_run

        # Set loss function
        if self.model.get_name() == "Autoencoder":
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_decoder = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.decoder_loss_weight = cfg.model.loss_weights.decoder

        # Raise error if cfg doesnt have 'num_classes' attribute
        self.num_classes = cfg.num_classes

        # Define a dict containing "task" and "num_classes" for intializing the metrics
        self.task_kwargs = {"num_classes": self.num_classes}

        # Creates and fills a dict to store all metrics
        self.metrics = nn.ModuleDict()
        self.metrics.update(self._get_metrics())
        for metric in self.metrics.values():
            metric.to(device)

    def _get_metrics(self) -> dict:
        """
        Build the dictionary of evaluation metrics.

        Regression metrics:
            - MSE
            - MAE

        Multiclass metrics:
            - Accuracy (micro-averaged)
            - F1, AUROC (macro-averaged)
            - Confusion Matrix

        Returns:
            dict: Dictionary mapping metric names to torchmetrics instances.
        """
        # Metrics for multiclass classification and regression
        macro_kwargs = {**self.task_kwargs, "average": "macro"}
        micro_kwargs = {**self.task_kwargs, "average": "micro"}
        metrics = {
            'conf_matrix': MulticlassConfusionMatrix(**self.task_kwargs),
            'accuracy_evaluation': MulticlassAccuracy(**micro_kwargs),
            'auroc_macro': MulticlassAUROC(**macro_kwargs),
            'f1_macro': MulticlassF1Score(**macro_kwargs),
            'mse_decoder': MeanSquaredError(),
            'mae_decoder': MeanAbsoluteError()
        }

        return metrics

    def _reset_metrics(self) -> None:
        """
        Resets all metric states. Called at the beginning of eval.
        """
        for metric in self.metrics.values():
            metric.reset()

    @torch.no_grad()
    def eval(self, return_metrics: bool = False) -> None:
        """
        Args:
            return_metrics: Optionally return metric. Default: False

        Run a full evaluation loop over the evaluation dataset and logs the metrics to
        Weights & Biases.
        """
        self.model.eval()
        self._reset_metrics()

        running_loss = 0.0

        for _, (x, y_true) in enumerate(self.eval_loader):
            x, y_true = x.to(self.device), y_true.to(self.device)

            # Make a prediction
            if self.model.get_name() == "Autoencoder":
                y_pred_classification, y_pred_decoder = self.model(x)

                y_pred_classification_probs = torch.softmax(y_pred_classification, dim=1)
                y_pred_classification_labels = torch.argmax(y_pred_classification, dim=1)

                loss_classification = self.criterion_classification(y_pred_classification, y_true)
                loss_decoder = self.criterion_decoder(y_pred_decoder, x)

                loss = self.classification_loss_weight * loss_classification + self.decoder_loss_weight * loss_decoder
            else:
                raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

            running_loss += loss.item()

            # Iterate through the dict to update the metrics
            for metric_name, metric in self.metrics.items():
                # Multiclass metrics
                if isinstance(metric, MulticlassAUROC):
                    # auroc uses probability instead of labels
                    metric.update(y_pred_classification_probs, y_true)
                elif isinstance(metric, MulticlassConfusionMatrix) or isinstance(metric, MulticlassAccuracy) or isinstance(metric, MulticlassF1Score):
                    metric.update(y_pred_classification_labels, y_true)
                # Regression metrics
                elif isinstance(metric, MeanSquaredError) or isinstance(metric, MeanAbsoluteError):
                    metric.update(y_pred_decoder, x)
                else:
                    logging.error(f"Metric {metric_name} not implemented in evaluation loop.")

        # Compute final metrics for hole dataset after iterating through all batches
        final_metrics = {}

        # Calculate average loss per batch
        final_metrics = {
            'total_loss_evaluation': running_loss / len(self.eval_loader)}

        # Compute all other metrics
        for metric_name, metric in self.metrics.items():
            try:
                final_metrics[metric_name] = metric.compute()
            except Exception as e:
                logging.warning("[Warning] Could not compute metric '%s'. Set to None. Error: %s",
                                metric_name, e)
                final_metrics[metric_name] = None
        print(f"Evaluation accuracy: {final_metrics['accuracy_evaluation'].item()}")


        if self.wb_run is not None:
            self.wb_run.log(final_metrics)

        if return_metrics:
            return final_metrics

## Trainer class
Class for handling the training of the model

TODO: Explain trainings loop

In [ ]:
class Trainer:
    '''
    Class to handle the training loop for a model.
    '''

    def __init__(
        self,
        cfg: any, # Contains the configuration parameters for training, such as learning rate, epochs, etc.
        train_loader: DataLoader,
        model: nn.Module,
        evaluator: Evaluator,
        device: torch.device,
        wb_run: Any
    ):
        '''
        Args:
            cfg (object): Configuration object with training parameters.
            train_loader (DataLoader): DataLoader for the training dataset.
            model (nn.Module): The model to be trained.
            evaluator (Evaluator): Evaluator for validation during training.
            device (torch.device): Device to run the training on.

        Initializes the Trainer with the given arguments.
        '''
        self.train_loader = train_loader

        # Ogranize training parameters, so not all are attributes of this class
        self.training_parameters = TrainingParameters(
            lr=cfg.training.learning_rate,
            lr_min=cfg.training.learning_rate_min,
            lr_scheduler=cfg.training.learning_rate_scheduler,
            beta1=cfg.training.beta1,
            beta2=cfg.training.beta2,
            weight_decay=cfg.training.weight_decay,
            epochs=cfg.training.num_epochs,
        )

        # Create optimizer
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=self.training_parameters.get_learning_rate(),
            betas=self.training_parameters.get_betas(),
            weight_decay=self.training_parameters.get_weight_decay()
        )

        # Initialize training components
        model.to(device)
        self.model = model
        self.optimizer = optimizer
        self.evaluator = evaluator
        self.device = device

        if cfg.dataset.data_augmentation.mixup.active:
            self.mixup = MixUp(alpha=cfg.dataset.data_augmentation.mixup.alpha)
        else:
            self.mixup = None

        # Set intervals
        self.log_interval = cfg.logging.log_interval
        self.eval_interval = cfg.logging.eval_interval

        # Set loss function
        if self.model.get_name() == "Autoencoder":
            self.criterion_classification = torch.nn.CrossEntropyLoss()
            self.criterion_decoder = torch.nn.MSELoss()
            self.classification_loss_weight = cfg.model.loss_weights.classification
            self.decoder_loss_weight = cfg.model.loss_weights.decoder

        # Change learning rate scheduler based on config
        self.scheduler = None
        if self.training_parameters.lr_scheduler == "cosine":
            self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.training_parameters.get_epochs(),
                eta_min=self.training_parameters.get_learning_rate_min()
            )

        self.wb_run = wb_run
        self.cfg = cfg


    def get_trained_model(self) -> nn.Module:
        '''
        Returns:
            The trained model

        Function to access the trained model by the Trainer class.
        '''
        return self.model

    def train(self):
        '''
        Main training loop.
        '''

        #TODO: Adjust for Autoencoder, training (freeze after some epoch...)
        for epoch in range(self.training_parameters.get_epochs()):
            start_time = time.time()
            # Set model to training mode
            self.model.train()
            running_loss = 0
            correct = 0
            total = 0

            for step, (x, y_true) in enumerate(self.train_loader):
                x, y_true = x.to(self.device), y_true.to(self.device)

                if self.mixup:
                    x, y_true = self.mixup(x, y_true)

                self.optimizer.zero_grad()

                if self.model.get_name() == "Autoencoder":
                    y_pred_classification, y_pred_decoder = self.model(x)

                    loss_classification = self.criterion_classification(y_pred_classification, y_true)
                    loss_decoder = self.criterion_decoder(y_pred_decoder, x)

                    loss = self.classification_loss_weight * loss_classification + self.decoder_loss_weight * loss_decoder
                else:
                    raise NotImplementedError(f"Model {self.model.get_name()} not implemented in training loop.")

                loss.backward()

                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                self.optimizer.step()

                running_loss += loss.item() * x.size(0)

                predicted_classes = torch.argmax(y_pred_classification, dim=1)
                total += y_true.size(0)
                correct += (predicted_classes == y_true).sum().item()


                if self.wb_run is not None and ((step + 1) % self.log_interval == 0):
                    accuracy = correct / total if total > 0 else 0
                    # Log the base learning rate (first param group, likely head)
                    current_lr = self.optimizer.param_groups[0]['lr']

                    self.wb_run.log({
                        "floor_accuracy_training": accuracy,
                        "total_loss_training": loss.item(),
                        "classification_loss_training": loss_classification.item(),
                        "learning_rate": current_lr
                    })

            end_time = time.time()
            if self.wb_run is not None:
                self.wb_run.log({"epoch_duration": end_time - start_time})

            if self.scheduler is not None:
                self.scheduler.step()

            print("Epoch [{0}/{1}]".format(epoch+1, self.training_parameters.get_epochs()))

            if (epoch + 1) % self.eval_interval == 0:
                # Set model to evaluation mode
                self.model.eval()
                with torch.no_grad():
                    self.evaluator.eval()

        logging.info("Training finished")
        # For optimization the metrics have to be returned
        return_metrics = self.cfg.optimization.active
        if return_metrics:
            return self.evaluator.eval(return_metrics)

## Model
Class of the selected model including the definition of the architecture as well as the forward porapagation

TODO: Explain model architecture & why (link to reference papers etc)

### Autoencoder
Autoencoder model 

In [ ]:
class Encoder(nn.Module):
    '''
    Encoder for the Autoencoder architecture.
    '''
    def __init__(self, input_dim: int, latent_dim: int, cfg: Any):
        super(Encoder, self).__init__()
        # TODO

    def forward(self, x):
        # TODO
        z = nn.Identity()(x)
        return z


class Decoder(nn.Module):
    '''
    Decoder for the Autoencoder architecture.
    '''
    def __init__(self, latent_dim: int, output_dim: int, cfg: Any):
        super(Decoder, self).__init__()
        # TODO

    def forward(self, z):
        # TODO
        x = nn.Sigmoid()(z)
        return x


class Classifier(nn.Module):
    '''
    Classifier for the Autoencoder architecture.
    '''
    def __init__(self, latent_dim: int, num_classes: int, cfg: Any):
        super(Classifier, self).__init__()
        # TODO

    def forward(self, z):
        # TODO
        y = nn.Sigmoid()(z)
        return y
    

class Autoencoder(nn.Module):
    '''
    Autoencoder architecture for the crop recommendation task.
    Consists of an Encoder, Decoder and Classifier.
    '''
    def __init__(self, cfg: Any):
        super(Autoencoder, self).__init__()
        num_classes = cfg.dataset.num_classes
        input_dim = self.cfg.input_dim
        latent_dim = self.cfg.autoencoder.latent_dim
        self.cfg = cfg.model
        

        self.encoder = Encoder(input_dim, latent_dim, self.cfg.encoder)
        self.decoder = Decoder(latent_dim, input_dim, self.cfg.decoder)
        self.classifier = Classifier(latent_dim, num_classes, self.cfg.classifier)

    def forward(self, x):
        z = self.encoder(x)
        x_reconstructed = self.decoder(z)
        y_classification = self.classifier(z)
        return y_classification, x_reconstructed

## Main function

TODO: Explain what happens in main function

In [ ]:
def main():
    '''
    Main function to run the training and evaluation pipeline.
    '''
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Create utility instance
    util = Utilities()

    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    _, train_dataloader = util.get_dataset(cfg, split='training')
    _, eval_dataloader = util.get_dataset(cfg, split='validation')
    _, test_dataloader = util.get_dataset(cfg, split='test')

    # Get Model
    if cfg.model.name == "Autoencoder":
        #TODO: Add Autoencoder Architecture class
        pass
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")

    model_summary = util.get_model_summary(model)
    print(model_summary)

    # Start a new wandb run to track this script
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_{timestamp}"
        else:
            run_name = f"{model.get_name()}_{timestamp}"
        wb_run = wandb.init(
            # Set the wandb entity where your project will be logged
            entity=cfg.logging.team_name,
            # Set the wandb project where this run will be logged
            project=cfg.logging.project_name,
            # Name of the run
            name=run_name,
            # Track hyperparameters and run metadata
            config={
                    "architecture": model.get_name(),
                    "dataset": cfg.dataset.name,
                    "training settings": {**dict(cfg.training)},
                    # Log all model config settings
                    "model settings": {**dict(cfg.model)},
                    "model architecture": model_summary,
                    "dataaugmentation settings": {**dict(cfg.dataset.dataaugmentation)},
                }
        )
    else:
        wb_run = None


    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Get trainer
    trainer = Trainer(cfg,
                      train_dataloader,
                      model,
                      evaluator,
                      device,
                      wb_run)

    # Train the model and perform final evaluation on test set
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    print("Running Evaluation")
    test_evaluator.eval()

    print("Finished")
#TODO: Calculate the meter out of the 0...1 prediction coordinates



## Bayesian optimizer
To optimize the hyperparameters of the model and achieve better performance

TODO: Explain bayesian optimizer, how done with AX platform and why we need/want it

In [ ]:
def optimize():
    '''
    Main function to run the training and evaluation pipeline.
    '''
    cfg = CONFIG
    if cfg is None:
        raise ValueError("Config dictionary must be provided.")

    # Create utility instance
    util = Utilities()

    # Set seed for reproducibility
    util.set_seed(cfg.seed)

    # Select device (GPU if available, else CPU)
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    logging.info('Using device: %s', device)

    # Get Datasets
    scaler = MinMaxScaler()
    _, train_dataloader, scaler = util.get_dataset(cfg, split='training', scaler=scaler)
    _, eval_dataloader, _ = util.get_dataset(cfg, split='validation', scaler=scaler)
    _, test_dataloader, _ = util.get_dataset(cfg, split='test', scaler=scaler)


    # Initialize AxClient for hyperparameter optimization
    ax_client = AxClient()

    ax_objectives = {
        name: ObjectiveProperties(
            minimize=props["minimize"], 
            threshold=props.get("threshold")
        )
        for name, props in cfg.optimization.optimization_config.objectives.items()
    }

    # Create experiment with data from optimization configuration
    ax_client.create_experiment(
        name="bayesian_optimization",
        parameters=cfg.optimization.search_space,
        objectives=ax_objectives,
        overwrite_existing_experiment=True
    )

    # Bayesian Optimization Loop
    for i in range(cfg.optimization.optimization_config.total_trials):
        print(f"Starting Trial {i + 1}/{cfg.optimization.optimization_config.total_trials}...")
        
        # Ax schlägt die nächsten optimalen Parameter vor (Bayesian State)
        parameters, trial_index = ax_client.get_next_trial()
        
        # Adapt parameters in the config for the current trial
        for param_name, param_value in parameters.items():
            util.set_by_path(cfg, param_name, param_value)

        # Get Model with adapted parameters
        if cfg.model.name == "Autoencoder":
            model = ...
        else:
            raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")
        
        model_summary = util.get_model_summary(model)
        logging.info(model_summary)

        # Start a new wandb run to track this script
        if cfg.logging.use_wandb and cfg.optimization.wb_logging:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M')
            wandb.login(key=cfg.logging.api_key)
            additional = cfg.logging.run_name_additional
            if additional != 'None' and additional is not None:
                run_name = f"{model.get_name()}_{additional}_optimizationTrial{i + 1}_{timestamp}"
            else:
                run_name = f"{model.get_name()}_optimizationTrial{i + 1}_{timestamp}"
            wb_run = wandb.init(
                # Set the wandb entity where your project will be logged
                entity=cfg.logging.team_name,
                # Set the wandb project where this run will be logged
                project=cfg.logging.project_name,
                # Name of the run
                name=run_name,
                # Track hyperparameters and run metadata
                config={
                    "architecture": model.get_name(),
                    "dataset": cfg.dataset.name,
                    "training settings": {**dict(cfg.training)},
                    # Log all model config settings
                    "model settings": {**dict(cfg.model)},
                    "model architecture": model_summary,
                    "dataaugmentation settings": {**dict(cfg.dataset.data_augmentation)},
                }
            )
        else:
            wb_run = None

        # Get evaluator
        evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

        # Get trainer
        trainer = Trainer(cfg,
                        train_dataloader,
                        model,
                        evaluator,
                        device,
                        wb_run)
        
        # Train the model and perform evaluation on validation set
        trainer.train()
        metrics = evaluator.eval(return_metrics=True)

        raw_data = {"classification_accuracy": metrics['accuracy_evaluation'].item()}
        print(raw_data)
        
        # Return results to ax
        ax_client.complete_trial(trial_index=trial_index, raw_data=raw_data)

        # Clean up
        del model
        del trainer
        del evaluator
        wb_run.finish() if wb_run is not None else None
        del wb_run
        gc.collect()

    # Retrieve best result
    best_params = list(ax_client.get_best_parameters())[0]
    print("\n--- Optimization complete ---")
    print(f"Best parameters: {best_params[0]}")
    print(f"Expected performance: {best_params[1]}")
    print("\n\n")
    print("Training & evaluating best model on test set")

    # Final evaluation on test set with the best found parameters

    # Adapt parameters in the config for the current trial
    for param_name, param_value in best_params[0].items():
        util.set_by_path(cfg, param_name, param_value)
    
    # Write configuration into a file
    util.save_best_parameters(cfg, filename="best_parameters.yaml")

    # Get Model with adapted parameters
    if cfg.model.name == "Ensemble_MLP":
        model = Ensemble_MLP(cfg.model)
    elif cfg.model.name == "Autoencoder":
        model = AutoEncoder(cfg.model)
    else:
        raise NotImplementedError(f"Model {cfg.model.name} not implemented in training loop.")

    # Start a new wandb run to track this script
    if cfg.logging.use_wandb:
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        wandb.login(key=cfg.logging.api_key)
        additional = cfg.logging.run_name_additional
        if additional != 'None' and additional is not None:
            run_name = f"{model.get_name()}_{additional}_bestResult_{timestamp}"
        else:
            run_name = f"{model.get_name()}_bestResult_{timestamp}"
        wb_run = wandb.init(
            # Set the wandb entity where your project will be logged
            entity=cfg.logging.team_name,
            # Set the wandb project where this run will be logged
            project=cfg.logging.project_name,
            # Name of the run
            name=run_name,
            # Track hyperparameters and run metadata
            config={
                "architecture": model.get_name(),
                "dataset": "UjiIndoorLoc",
                "training settings": {**dict(cfg.training)},
                # Log all model config settings
                "model settings": {**dict(cfg.model)},
                "model architecture": model_summary,
                "dataaugmentation settings": {**dict(cfg.dataset.data_augmentation)},
            }
        )
    else:
        wb_run = None
    
    # Get evaluator
    evaluator = Evaluator(cfg, eval_dataloader, model, device, wb_run)

    # Get trainer
    trainer = Trainer(cfg,
                    train_dataloader,
                    model,
                    evaluator,
                    device,
                    wb_run)
    
    trainer.train()

    test_evaluator = Evaluator(cfg, test_dataloader, trainer.get_trained_model(), device, wb_run)
    logging.info("Running Evaluation")
    test_evaluator.eval()


In [ ]:
if __name__ == '__main__':
    if CONFIG.optimization.active:
        optimize()
    else:
        main()